# Semantic MEP–Space Coordination Reasoner

## Semantic clash detection beyond geometry across many IFC files

This notebook demonstrates a TopologicPy workflow that loads **all IFC files in a folder**, builds one `TGraph` per IFC file, classifies each graph as architecture, MEP, or mixed, and then synchronises them into a coordinated semantic graph.

The goal is not only to detect geometric clashes, but to reason about whether MEP objects are **spatially, semantically, and systemically coordinated** with the architectural model.

The workflow combines:

- IFC semantics from multiple discipline/model files;
- TopologicPy `TGraph` objects, one per IFC;
- provenance-preserving graph federation;
- Brick classes for MEP elements;
- BOT-compatible spatial containment predicates;
- TopologicPy ontology predicates such as `top:connectsPort`, `top:hasOpening`, `top:fillsOpening`, `top:hasPropertySet`, `top:hasMaterial`, and `top:servesSpatialStructure`;
- inferred coordination issues such as missing penetrations, unlocated terminals, and overasserted flow direction.

The important modelling principle is:

- vertex `ontology_class` expresses **what an entity is**;
- edge `ontology_predicate` expresses **what a relationship means**;
- graph-level provenance records **which IFC file supplied the assertion**.


## 1. Configure the IFC folder

Place all source IFC files in one folder. The folder may contain one architectural model, several architectural models, one MEP model, several MEP models, or mixed discipline IFC files.

The notebook discovers all `.ifc` files recursively, imports each file into its own `TGraph`, and then federates them into one coordinated graph.


In [1]:
import sys
from pathlib import Path
sys.path.append("C:/Users/sarwj/OneDrive - Cardiff University/Documents/GitHub/topologicpy/src")


# Folder containing all source IFC files.
IFC_FOLDER = Path(r"C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC")

# Recursive glob pattern. Use "*.ifc" for one folder only, or "**/*.ifc" for nested folders.
IFC_GLOB = "**/*.ifc"

# Optional manual role overrides by file stem or file name.
# Valid roles: "architecture", "mep", "mixed", "unknown".
# Example:
# IFC_ROLE_OVERRIDES = {
#     "Architecture": "architecture",
#     "MEP": "mep",
# }
IFC_ROLE_OVERRIDES = {}

# Optional output files.
OUT_DIR = Path("mep_space_coordination_outputs")
OUT_DIR.mkdir(exist_ok=True)

IFC_PATHS = sorted(p for p in IFC_FOLDER.glob(IFC_GLOB) if p.is_file() and p.suffix.lower() == ".ifc")

print("IFC folder:", IFC_FOLDER.resolve())
print("IFC files found:", len(IFC_PATHS))
for p in IFC_PATHS:
    print(" -", p)
print("Output folder:", OUT_DIR.resolve())

if not IFC_PATHS:
    raise FileNotFoundError(
        f"No IFC files found in {IFC_FOLDER.resolve()} using pattern {IFC_GLOB!r}. "
        "Update IFC_FOLDER/IFC_GLOB or place IFC files in that folder."
    )


IFC folder: C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC
IFC files found: 5
 - C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC\Ifc2s3_Duplex_Electrical.ifc
 - C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC\Ifc2x3_Duplex_Architecture.ifc
 - C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC\Ifc2x3_Duplex_Mechanical.ifc
 - C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC\Ifc2x3_Duplex_MEP.ifc
 - C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\assets\IFC\Ifc2x3_Duplex_Plumbing.ifc
Output folder: C:\Users\sarwj\OneDrive - Cardiff University\Documents\GitHub\topologicpy\notebooks\mep_space_coordination_outputs


## 2. Import TopologicPy classes

This notebook assumes you have updated `IFC.py`, `TGraph.py`, `KnowledgeGraph.py`, and `topologicpy.ttl` with the recent Brick/MEP and edge-predicate additions.

Important expected dictionary keys:

- vertices: `ifc_type`, `ontology_class`, `brick_class`, `bot_class`, `global_id`, `name`, `label`;
- edges: `ifc_type` or `ifc_relationship`, `ontology_class`, `ontology_predicate`, `inverse_predicate`, `relationship`.

In [2]:
import math
import re
import json
from collections import Counter, defaultdict
from pprint import pprint

try:
    from topologicpy.IFC import IFC
    from topologicpy.TGraph import TGraph
    from topologicpy.KnowledgeGraph import KnowledgeGraph
except Exception:
    # Local-development fallback if running this notebook beside the source files.
    from IFC import IFC
    from TGraph import TGraph
    from KnowledgeGraph import KnowledgeGraph

print("Imported IFC, TGraph, and KnowledgeGraph.")

Imported IFC, TGraph, and KnowledgeGraph.


## 3. Load all IFC files as separate TGraphs

`IFC.TGraphByPath` keeps IFC import logic on the IFC side while still returning a graph suitable for TopologicPy analysis and RDF conversion.

For this use case, `dictionaryMode="full"` is preferred because property sets, type assignments, material associations, classifications, and relationship metadata are useful for reasoning.

Each graph is tagged with:

- `source_path`
- `source_file`
- `source_stem`
- inferred `source_role`: `architecture`, `mep`, `mixed`, or `unknown`


In [3]:
def _record_dict(record):
    return record.get("dictionary", {}) or {}

def _norm(value):
    return "" if value is None else str(value).strip().lower()

def _ifc_type_from_dict(d):
    return d.get("ifc_type") or d.get("IFCType") or d.get("type") or ""

def _class_values_from_dict(d):
    return " ".join(str(d.get(k, "")) for k in [
        "ontology_class", "brick_class", "bot_class", "topologic_class", "ifc_type"
    ]).lower()

def classify_graph_role(graph, path=None):
    """Heuristically classify a source IFC graph by its emitted ontology/IFC classes."""
    file_keys = []
    if path is not None:
        path = Path(path)
        file_keys = [path.name, path.stem]
        for key in file_keys:
            if key in IFC_ROLE_OVERRIDES:
                return IFC_ROLE_OVERRIDES[key]

    arch_hits = 0
    mep_hits = 0
    spatial_hits = 0

    mep_tokens = [
        "brick:", "duct", "pipe", "cable", "terminal", "luminaire", "outlet",
        "pump", "fan", "boiler", "chiller", "valve", "damper", "sensor",
        "meter", "controller", "port", "flowsystem", "distribution", "mep"
    ]
    arch_tokens = [
        "top:wall", "top:slab", "top:door", "top:window", "top:opening", "top:stair",
        "top:railing", "top:column", "top:building", "top:storey", "top:site",
        "ifcwall", "ifcslab", "ifcdoor", "ifcwindow", "ifcopening", "ifcstair"
    ]
    spatial_tokens = ["top:space", "ifcspace", "top:zone", "ifczone"]

    for v in getattr(graph, "_vertices", []):
        if not v.get("active", True):
            continue
        text = _class_values_from_dict(_record_dict(v))
        if any(t in text for t in mep_tokens):
            mep_hits += 1
        if any(t in text for t in arch_tokens):
            arch_hits += 1
        if any(t in text for t in spatial_tokens):
            spatial_hits += 1

    # Spatial hits are architectural context, but mixed models are common.
    arch_score = arch_hits + spatial_hits
    mep_score = mep_hits

    if mep_score > 0 and arch_score > 0:
        return "mixed"
    if mep_score > 0:
        return "mep"
    if arch_score > 0:
        return "architecture"
    return "unknown"

def load_ifc_tgraph(path, label=None, includeTypes=None, excludeTypes=None, includeRels=None, excludeRels=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"IFC path does not exist: {path}")

    label = label or path.stem
    graph = IFC.TGraphByPath(
        str(path),
        importMode="topology",
        dictionaryMode="full",
        includeTypes=includeTypes,
        excludeTypes=excludeTypes,
        includeRels=includeRels,
        excludeRels=excludeRels,
        storeBREP=False,
        useInternalVertex=True,
        silent=False,
    )
    if graph is None:
        raise RuntimeError(f"Could not import IFC as TGraph: {path}")

    if not isinstance(graph._dictionary, dict):
        graph._dictionary = {}

    role = classify_graph_role(graph, path)
    graph._dictionary.update({
        "label": label,
        "source_path": str(path),
        "source_file": path.name,
        "source_stem": path.stem,
        "source_role": role,
    })
    return graph

graph_records = []
for i, path in enumerate(IFC_PATHS):
    label = path.stem
    graph = load_ifc_tgraph(path, label=label)
    record = {
        "graph_index": i,
        "label": label,
        "path": path,
        "role": graph._dictionary.get("source_role", "unknown"),
        "graph": graph,
    }
    graph_records.append(record)
    print(f"[{i}] {label}: role={record['role']} vertices={TGraph.Order(graph)} edges={TGraph.Size(graph)}")

ifc_graphs = [r["graph"] for r in graph_records]
architecture_graphs = [r for r in graph_records if r["role"] in {"architecture", "mixed"}]
mep_graphs = [r for r in graph_records if r["role"] in {"mep", "mixed"}]

print("\nArchitecture/mixed source graphs:", len(architecture_graphs))
print("MEP/mixed source graphs:", len(mep_graphs))


[0] Ifc2s3_Duplex_Electrical: role=mixed vertices=1783 edges=1440
[1] Ifc2x3_Duplex_Architecture: role=architecture vertices=3985 edges=3151
[2] Ifc2x3_Duplex_Mechanical: role=mixed vertices=7590 edges=6197
[3] Ifc2x3_Duplex_MEP: role=mixed vertices=14998 edges=11643
[4] Ifc2x3_Duplex_Plumbing: role=mixed vertices=12244 edges=8750

Architecture/mixed source graphs: 5
MEP/mixed source graphs: 4


## 4. Inspect ontology classes and semantic edge predicates

This is a first validation step. MEP graphs should contain Brick classes on MEP vertices and meaningful `ontology_predicate` values on relationship edges.

A critical expected correction is:

```text
IfcRelConnectsPorts → top:connectsPort
```

not `brick:feeds`, because an IFC port connection does not necessarily encode flow direction.


In [4]:
def vertex_class_summary(graph, top_n=50):
    c = Counter()
    examples = defaultdict(list)
    for v in graph._vertices:
        if not v.get("active", True):
            continue
        d = v.get("dictionary", {}) or {}
        cls = d.get("ontology_class", "<missing>")
        c[cls] += 1
        if len(examples[cls]) < 3:
            examples[cls].append(d.get("ifc_type") or d.get("label") or d.get("name") or v.get("index"))
    rows = []
    for cls, n in c.most_common(top_n):
        rows.append({"ontology_class": cls, "count": n, "examples": examples[cls]})
    return rows

def edge_predicate_summary(graph, top_n=50):
    c = Counter()
    examples = defaultdict(list)
    for e in graph._edges:
        if not e.get("active", True):
            continue
        d = e.get("dictionary", {}) or {}
        pred = d.get("ontology_predicate", "<missing>")
        c[pred] += 1
        if len(examples[pred]) < 3:
            examples[pred].append(d.get("ifc_relationship") or d.get("ifc_type") or d.get("relationship") or e.get("index"))
    rows = []
    for pred, n in c.most_common(top_n):
        rows.append({"ontology_predicate": pred, "count": n, "examples": examples[pred]})
    return rows

for rec in graph_records:
    print("\n===", rec["label"], f"({rec['role']})", "===")
    print("Vertex ontology classes:")
    pprint(vertex_class_summary(rec["graph"], 20))
    print("Edge ontology predicates:")
    pprint(edge_predicate_summary(rec["graph"], 20))



=== Ifc2s3_Duplex_Electrical (mixed) ===
Vertex ontology classes:
[{'count': 838,
  'examples': [910, 911, 912],
  'ontology_class': 'top:Relationship'},
 {'count': 783,
  'examples': [127, 128, 129],
  'ontology_class': 'top:PropertySet'},
 {'count': 82,
  'examples': [21, 22, 23],
  'ontology_class': 'brick:Terminal_Unit'},
 {'count': 16,
  'examples': [1752, 1753, 1754],
  'ontology_class': 'top:ClassificationReference'},
 {'count': 14, 'examples': [7, 8, 9], 'ontology_class': 'brick:Equipment'},
 {'count': 14,
  'examples': [1768, 1769, 1771],
  'ontology_class': 'top:MaterialSet'},
 {'count': 6,
  'examples': [113, 114, 118],
  'ontology_class': 'brick:Electrical_Equipment'},
 {'count': 4, 'examples': [0, 1, 2], 'ontology_class': 'top:Element'},
 {'count': 4, 'examples': [944, 945, 946], 'ontology_class': 'top:Interface'},
 {'count': 3,
  'examples': [4, 5, 6],
  'ontology_class': 'brick:Control_Equipment'},
 {'count': 3, 'examples': [104, 105, 106], 'ontology_class': 'top:Storey

## 5. Helper functions for graph records, semantics, and geometry

The coordination checks below use two levels of evidence:

1. **semantic evidence**, from IFC/Brick/BOT/TopologicPy dictionaries;
2. **geometric evidence**, from Topologic representations where available, falling back to AABBs and coordinates.

This keeps the notebook robust across different IFC authoring/export pipelines.

In [5]:
def active_vertices(graph):
    return [v for v in graph._vertices if v.get("active", True)]

def active_edges(graph):
    return [e for e in graph._edges if e.get("active", True)]

def vd(v):
    return v.get("dictionary", {}) or {}

def ed(e):
    return e.get("dictionary", {}) or {}

def qname_local(qname):
    if qname is None:
        return ""
    return str(qname).split(":")[-1]

def norm_text(value):
    return "" if value is None else str(value).strip().lower()

def element_name(record):
    d = record.get("dictionary", {}) or {}
    return (
        d.get("label") or d.get("name") or d.get("Name") or d.get("global_id") or
        d.get("ifc_guid") or d.get("ifc_id") or record.get("index")
    )

def is_class(d, candidates):
    classes = {norm_text(d.get(k)) for k in ("ontology_class", "brick_class", "bot_class", "topologic_class")}
    if isinstance(candidates, str):
        candidates = [candidates]
    return any(norm_text(c) in classes for c in candidates)

def ifc_type(d):
    return d.get("ifc_type") or d.get("IFCType") or d.get("ifc_class") or d.get("ifcClass")

def is_arch_space(v):
    d = vd(v)
    return is_class(d, ["top:Space", "bot:Space"]) or norm_text(ifc_type(d)) == "ifcspace"

def is_storey(v):
    d = vd(v)
    return is_class(d, ["top:Storey", "bot:Storey"]) or norm_text(ifc_type(d)) == "ifcbuildingstorey"

def is_wall_or_slab(v):
    d = vd(v)
    cls = norm_text(d.get("ontology_class"))
    t = norm_text(ifc_type(d))
    return cls in {"top:wall", "top:slab"} or t in {"ifcwall", "ifcwallstandardcase", "ifcslab", "ifcslabstandardcase"}

def is_opening(v):
    d = vd(v)
    return is_class(d, "top:Opening") or norm_text(ifc_type(d)) == "ifcopeningelement"

def is_mep_distribution_or_equipment(v):
    d = vd(v)
    oc = norm_text(d.get("ontology_class"))
    bc = norm_text(d.get("brick_class"))
    t = norm_text(ifc_type(d))
    return oc.startswith("brick:") or bc.startswith("brick:") or any(token in t for token in ["duct", "pipe", "cable", "flow", "terminal", "fixture", "port", "system"])

def is_terminal(v):
    d = vd(v)
    oc = norm_text(d.get("ontology_class"))
    t = norm_text(ifc_type(d))
    return oc in {"brick:terminal_unit", "brick:air_terminal", "brick:plumbing_fixture", "brick:luminaire", "brick:outlet"} or "terminal" in t or "fixture" in t or "outlet" in t

def is_sanitary_or_plumbing(v):
    d = vd(v)
    oc = norm_text(d.get("ontology_class"))
    t = norm_text(ifc_type(d))
    return oc == "brick:plumbing_fixture" or "sanitary" in t or "waste" in t

def is_space_wet_or_service(space_v):
    d = vd(space_v)
    text = " ".join(str(d.get(k, "")) for k in ["label", "name", "Name", "long_name", "description", "room_type", "space_type"])
    text = text.lower()
    wet_terms = ["wc", "toilet", "bath", "bathroom", "shower", "wash", "kitchen", "utility", "plant", "mechanical", "service", "janitor", "cleaner", "laundry"]
    return any(term in text for term in wet_terms)

## 6. Geometry helpers: representation, AABB, containment, and intersection

The strongest version of this use case uses actual topological containment/intersection. If exact operations fail or are unavailable, the notebook falls back to bounding boxes. This is suitable for early coordination screening; detailed clash validation can later use exact Topologic geometry operations.

In [6]:
def representation(record):
    return record.get("representation", None)

def coords_from_dict(d):
    for keys in [("x", "y", "z")]:
        if all(k in d for k in keys):
            try:
                return (float(d["x"]), float(d["y"]), float(d.get("z", 0.0)))
            except Exception:
                pass
    for key in ["coordinates", "coords", "xyz"]:
        c = d.get(key)
        if isinstance(c, (list, tuple)) and len(c) >= 2:
            try:
                return (float(c[0]), float(c[1]), float(c[2]) if len(c) > 2 else 0.0)
            except Exception:
                pass
    return None

def vertex_point(v):
    # Prefer dictionary coordinates, then TGraph.Coordinates if possible.
    c = coords_from_dict(vd(v))
    if c is not None:
        return c
    return None

def aabb_from_record(record):
    """Return (xmin, ymin, zmin, xmax, ymax, zmax) or None."""
    d = record.get("dictionary", {}) or {}
    # Direct AABB keys, common in some pipelines.
    possible = [
        ("xmin", "ymin", "zmin", "xmax", "ymax", "zmax"),
        ("min_x", "min_y", "min_z", "max_x", "max_y", "max_z"),
    ]
    for keys in possible:
        if all(k in d for k in keys):
            try:
                return tuple(float(d[k]) for k in keys)
            except Exception:
                pass
    # Try TopologicPy's bounding box if representation exists.
    topo = representation(record)
    if topo is not None:
        try:
            from topologicpy.Topology import Topology
            bb = Topology.BoundingBox(topo)
            bd = Topology.Dictionary(bb)
            # Try common dictionary conversion.
            try:
                from topologicpy.Dictionary import Dictionary
                bd = Dictionary.PythonDictionary(bd)
            except Exception:
                pass
            if isinstance(bd, dict):
                for keys in possible:
                    if all(k in bd for k in keys):
                        return tuple(float(bd[k]) for k in keys)
        except Exception:
            pass
    # Single point fallback.
    c = coords_from_dict(d)
    if c is not None:
        x, y, z = c
        return (x, y, z, x, y, z)
    return None

def aabb_overlap(a, b, tol=1e-6):
    if a is None or b is None:
        return None
    ax0, ay0, az0, ax1, ay1, az1 = a
    bx0, by0, bz0, bx1, by1, bz1 = b
    return not (ax1 < bx0 - tol or bx1 < ax0 - tol or ay1 < by0 - tol or by1 < ay0 - tol or az1 < bz0 - tol or bz1 < az0 - tol)

def point_in_aabb(p, a, tol=1e-6):
    if p is None or a is None:
        return None
    x, y, z = p
    xmin, ymin, zmin, xmax, ymax, zmax = a
    return (xmin - tol <= x <= xmax + tol and ymin - tol <= y <= ymax + tol and zmin - tol <= z <= zmax + tol)

def exact_intersects(record_a, record_b, tol=1e-4):
    ta, tb = representation(record_a), representation(record_b)
    if ta is None or tb is None:
        return None
    try:
        from topologicpy.Topology import Topology
        # Intersect returns a topology or None. Keep this deliberately generic.
        inter = Topology.Intersect(ta, tb, tolerance=tol)
        return inter is not None
    except Exception:
        return None

def probably_intersects(record_a, record_b, tol=1e-4):
    exact = exact_intersects(record_a, record_b, tol=tol)
    if exact is not None:
        return exact, "exact_topology"
    aa, bb = aabb_from_record(record_a), aabb_from_record(record_b)
    ov = aabb_overlap(aa, bb, tol=tol)
    if ov is not None:
        return ov, "aabb"
    return False, "unresolved"

def probably_contains(container_record, item_record, tol=1e-4):
    item_point = vertex_point(item_record)
    caabb = aabb_from_record(container_record)
    if item_point is not None and caabb is not None:
        return point_in_aabb(item_point, caabb, tol=tol), "point_in_aabb"
    # Fallback: item AABB center in container AABB.
    iaabb = aabb_from_record(item_record)
    if iaabb is not None and caabb is not None:
        p = ((iaabb[0]+iaabb[3])/2, (iaabb[1]+iaabb[4])/2, (iaabb[2]+iaabb[5])/2)
        return point_in_aabb(p, caabb, tol=tol), "aabb_center"
    return False, "unresolved"

## 7. Synchronise all source graphs

The synchronisation strategy is a **provenance-preserving coordinated union graph**. Vertices and edges from every source IFC graph are copied into a new `TGraph`, preserving:

- source file path;
- source file stem;
- inferred source role;
- original graph index;
- original vertex and edge indices.

Additional inferred coordination edges are added later. This keeps imported IFC facts separate from derived TopologicPy reasoning facts.


In [7]:
def copy_graph_into(source_graph, target_graph, source_record):
    index_map = {}
    source_label = source_record["label"]
    source_path = str(source_record["path"])
    source_role = source_record["role"]
    source_graph_index = source_record["graph_index"]

    for v in active_vertices(source_graph):
        d = dict(vd(v))
        d["source_model"] = source_label
        d["source_path"] = source_path
        d["source_file"] = Path(source_path).name
        d["source_role"] = source_role
        d["source_graph_index"] = source_graph_index
        d["source_vertex_index"] = v.get("index")
        new_idx = target_graph.AddVertex(dictionary=d, representation=v.get("representation"))
        index_map[v.get("index")] = new_idx

    for e in active_edges(source_graph):
        src = index_map.get(e.get("src"))
        dst = index_map.get(e.get("dst"))
        if src is None or dst is None:
            continue
        d = dict(ed(e))
        d["source_model"] = source_label
        d["source_path"] = source_path
        d["source_file"] = Path(source_path).name
        d["source_role"] = source_role
        d["source_graph_index"] = source_graph_index
        d["source_edge_index"] = e.get("index")
        target_graph.AddEdge(src, dst, directed=e.get("directed", False), dictionary=d, representation=e.get("representation"))
    return index_map

coord_graph = TGraph(directed=True, allowSelfLoops=True, allowParallelEdges=True, dictionary={
    "label": "Federated MEP–Space Coordination Graph",
    "ontology_class": "top:Graph",
    "use_case": "Semantic MEP coordination and clash detection beyond geometry",
    "source_count": len(graph_records),
    "source_files": [str(r["path"]) for r in graph_records],
})

source_maps = {}
for rec in graph_records:
    source_maps[rec["label"]] = copy_graph_into(rec["graph"], coord_graph, rec)

print("Coordination graph:", coord_graph)
print("Vertices/edges:", TGraph.Order(coord_graph), TGraph.Size(coord_graph))
print("Source maps:", {k: len(v) for k, v in source_maps.items()})


Coordination graph: TGraph(vertices=40600, edges=31181, directed)
Vertices/edges: 40600 31181
Source maps: {'Ifc2s3_Duplex_Electrical': 1783, 'Ifc2x3_Duplex_Architecture': 3985, 'Ifc2x3_Duplex_Mechanical': 7590, 'Ifc2x3_Duplex_MEP': 14998, 'Ifc2x3_Duplex_Plumbing': 12244}


## 8. Derive spatial location: MEP elements located in architectural spaces

This step asks:

> Which architectural space most likely contains each MEP element across all loaded IFC files?

The derived edge is represented as:

```python
ontology_class = "top:Relationship"
ontology_predicate = "top:locatedIn"
relationship = "located_in"
inferred = True
```

For RDF output this becomes:

```turtle
inst:MepElement top:locatedIn inst:Space .
```


In [8]:
coord_vertex_indices = [v["index"] for v in active_vertices(coord_graph)]

arch_space_indices = [i for i in coord_vertex_indices if is_arch_space(coord_graph._vertices[i])]
mep_element_indices = [i for i in coord_vertex_indices if is_mep_distribution_or_equipment(coord_graph._vertices[i])]

print("Architectural spaces across all source graphs:", len(arch_space_indices))
print("MEP elements/equipment/terminals across all source graphs:", len(mep_element_indices))

def add_inferred_edge(graph, src, dst, predicate, relationship, confidence=1.0, reason=None, evidence=None, inverse_predicate=None):
    d = {
        "ontology_class": "top:Relationship",
        "ontology_predicate": predicate,
        "relationship": relationship,
        "inferred": True,
        "confidence": float(confidence),
        "source": "TopologicPy coordination reasoner",
    }
    if inverse_predicate:
        d["inverse_predicate"] = inverse_predicate
    if reason:
        d["reason"] = reason
    if evidence:
        d["evidence"] = evidence
    return graph.AddEdge(src, dst, directed=True, dictionary=d)

located_edges = []
for mi in mep_element_indices:
    mep_v = coord_graph._vertices[mi]
    candidates = []
    for si in arch_space_indices:
        space_v = coord_graph._vertices[si]
        ok, method = probably_contains(space_v, mep_v)
        if ok:
            # Prefer smaller enclosing spaces when AABBs are available.
            a = aabb_from_record(space_v)
            volume = None
            if a:
                volume = max(0, a[3]-a[0]) * max(0, a[4]-a[1]) * max(0, a[5]-a[2])
            candidates.append((volume if volume is not None else float("inf"), si, method))
    if candidates:
        candidates.sort(key=lambda x: x[0])
        _, si, method = candidates[0]
        located_edges.append(add_inferred_edge(
            coord_graph, mi, si,
            predicate="top:locatedIn",
            inverse_predicate="top:containsElement",
            relationship="located_in",
            confidence=0.85 if method != "unresolved" else 0.4,
            reason="MEP element location derived from architectural space containment across federated IFC graphs.",
            evidence=method,
        ))

print("Derived top:locatedIn edges:", len(located_edges))


Architectural spaces across all source graphs: 102
MEP elements/equipment/terminals across all source graphs: 2659
Derived top:locatedIn edges: 0


## 9. Geometric intersections: MEP elements crossing walls, slabs, or openings

This step asks:

> Which MEP elements geometrically intersect architectural walls/slabs/openings across all loaded IFC files?

The reasoner records likely intersections with `top:intersects`.


In [9]:
arch_barrier_indices = [i for i in coord_vertex_indices if is_wall_or_slab(coord_graph._vertices[i])]
arch_opening_indices = [i for i in coord_vertex_indices if is_opening(coord_graph._vertices[i])]

print("Architectural barriers across all source graphs: walls/slabs =", len(arch_barrier_indices))
print("Architectural openings across all source graphs =", len(arch_opening_indices))

intersection_edges = []
for mi in mep_element_indices:
    mep_v = coord_graph._vertices[mi]
    for bi in arch_barrier_indices:
        if mi == bi:
            continue
        barrier_v = coord_graph._vertices[bi]
        ok, method = probably_intersects(mep_v, barrier_v)
        if ok:
            intersection_edges.append(add_inferred_edge(
                coord_graph, mi, bi,
                predicate="top:intersects",
                inverse_predicate="top:isIntersectedBy",
                relationship="intersects",
                confidence=0.95 if method == "exact_topology" else 0.65,
                reason="MEP element appears to intersect an architectural barrier across federated IFC graphs.",
                evidence=method,
            ))

print("Derived top:intersects edges:", len(intersection_edges))


Architectural barriers across all source graphs: walls/slabs = 78
Architectural openings across all source graphs = 50
Derived top:intersects edges: 8


## 10. Semantic coordination rules

The following rules are intentionally simple and explainable.

### Rule A — Missing penetration / missing opening

If a duct, pipe, cable carrier, or other MEP distribution element intersects a wall or slab, the intersection should normally be coordinated with an opening or penetration.

This goes beyond ordinary clash detection because the system distinguishes between:

- an unacceptable clash: `MEP element intersects wall/slab`;
- an acceptable penetration: `MEP element intersects wall/slab` **and** an opening exists at the crossing.

### Rule B — Terminal not located in a space

Air terminals, luminaires, outlets, plumbing fixtures, and other terminal devices should generally be locatable within an architectural space.

### Rule C — Sanitary terminal in non-wet/non-service space

A sanitary/plumbing fixture located in an office, corridor, or similar non-wet space may indicate incorrect placement, bad room naming, or model coordination errors.

### Rule D — IFC port connection is not automatically Brick flow

`IfcRelConnectsPorts` should produce `top:connectsPort`. `brick:feeds` should be inferred only when direction is supported by port flow direction, system metadata, or a known upstream/downstream path.

In [10]:
def issue_record(issue_type, severity, element_idx, message, **kwargs):
    v = coord_graph._vertices[element_idx]
    d = vd(v)
    row = {
        "issue_type": issue_type,
        "severity": severity,
        "element": element_name(v),
        "element_index": element_idx,
        "ifc_type": ifc_type(d),
        "ontology_class": d.get("ontology_class"),
        "brick_class": d.get("brick_class"),
        "message": message,
    }
    row.update(kwargs)
    return row

issues = []

# Rule A: MEP crossing wall/slab without nearby/intersecting opening.
def has_opening_for_crossing(mep_idx, barrier_idx):
    mep_v = coord_graph._vertices[mep_idx]
    barrier_v = coord_graph._vertices[barrier_idx]
    # Evidence: an opening intersects both the MEP element and the barrier, or overlaps both AABBs.
    for oi in arch_opening_indices:
        opening_v = coord_graph._vertices[oi]
        ok1, _ = probably_intersects(opening_v, barrier_v)
        if not ok1:
            continue
        ok2, _ = probably_intersects(opening_v, mep_v)
        if ok2:
            return True, oi
    return False, None

for edge_idx in intersection_edges:
    e = coord_graph._edges[edge_idx]
    mep_idx = e["src"]
    barrier_idx = e["dst"]
    mep_v = coord_graph._vertices[mep_idx]
    barrier_v = coord_graph._vertices[barrier_idx]
    mep_d = vd(mep_v)
    # Restrict to distribution-like things; terminals can touch ceilings/walls legitimately.
    cls_text = " ".join(str(mep_d.get(k, "")) for k in ["ontology_class", "brick_class", "ifc_type"]).lower()
    if not any(token in cls_text for token in ["duct", "pipe", "cable", "flowsegment", "flow_segment", "equipment"]):
        continue
    ok, opening_idx = has_opening_for_crossing(mep_idx, barrier_idx)
    if not ok:
        issues.append(issue_record(
            "missing_penetration_or_opening",
            "high",
            mep_idx,
            "MEP distribution element intersects an architectural barrier, but no matching opening was found.",
            architectural_element=element_name(barrier_v),
            architectural_index=barrier_idx,
            architectural_ifc_type=ifc_type(vd(barrier_v)),
        ))
        add_inferred_edge(coord_graph, mep_idx, barrier_idx,
                          predicate="top:hasMissingOpening",
                          relationship="has_missing_opening",
                          confidence=0.75,
                          reason="Intersects barrier without an opening that intersects both barrier and MEP element.")

# Build location lookup from inferred top:locatedIn edges.
located_in = defaultdict(list)
for e in active_edges(coord_graph):
    d = ed(e)
    if d.get("ontology_predicate") == "top:locatedIn":
        located_in[e["src"]].append(e["dst"])

# Rule B: terminals should be located in a space.
for mi in mep_element_indices:
    if not is_terminal(coord_graph._vertices[mi]):
        continue
    if mi not in located_in:
        issues.append(issue_record(
            "terminal_not_located_in_space",
            "medium",
            mi,
            "Terminal/equipment has no derived architectural space location.",
        ))

# Rule C: sanitary/plumbing fixture in non-wet space.
for mi, spaces in located_in.items():
    mep_v = coord_graph._vertices[mi]
    if not is_sanitary_or_plumbing(mep_v):
        continue
    for si in spaces:
        space_v = coord_graph._vertices[si]
        if not is_space_wet_or_service(space_v):
            issues.append(issue_record(
                "sanitary_terminal_in_non_wet_space",
                "medium",
                mi,
                "Sanitary/plumbing fixture appears to be located in a space that is not labelled as wet or service.",
                space=element_name(space_v),
                space_index=si,
                space_ifc_type=ifc_type(vd(space_v)),
            ))

# Rule D: direct imported IfcRelConnectsPorts edges should not claim brick:feeds.
for e in active_edges(coord_graph):
    d = ed(e)
    rel_text = norm_text(d.get("ifc_relationship") or d.get("ifc_type") or d.get("relationship"))
    pred = d.get("ontology_predicate")
    if "ifcrelconnectsports" in rel_text or rel_text == "connects_ports":
        if pred == "brick:feeds":
            issues.append(issue_record(
                "overasserted_flow_direction",
                "high",
                e["src"],
                "IfcRelConnectsPorts was mapped directly to brick:feeds. Use top:connectsPort unless flow direction is proven.",
                edge_index=e.get("index"),
                predicate=pred,
            ))

print("Coordination issues found:", len(issues))
for row in issues[:20]:
    pprint(row)

Coordination issues found: 711
{'architectural_element': 63,
 'architectural_ifc_type': None,
 'architectural_index': 1846,
 'brick_class': 'brick:Equipment',
 'element': 87,
 'element_index': 5855,
 'ifc_type': None,
 'issue_type': 'missing_penetration_or_opening',
 'message': 'MEP distribution element intersects an architectural barrier, but '
            'no matching opening was found.',
 'ontology_class': 'brick:Equipment',
 'severity': 'high'}
{'architectural_element': 63,
 'architectural_ifc_type': None,
 'architectural_index': 1846,
 'brick_class': 'brick:Equipment',
 'element': 122,
 'element_index': 13480,
 'ifc_type': None,
 'issue_type': 'missing_penetration_or_opening',
 'message': 'MEP distribution element intersects an architectural barrier, but '
            'no matching opening was found.',
 'ontology_class': 'brick:Equipment',
 'severity': 'high'}
{'architectural_element': 63,
 'architectural_ifc_type': None,
 'architectural_index': 1846,
 'brick_class': 'brick:Equipme

## 11. Summarise issues as a table

The table below is the main “semantic clash detection beyond geometry” result. It combines geometry, spatial containment, IFC relationship semantics, Brick classes, and TopologicPy inferred predicates.

In [11]:
try:
    import pandas as pd
    issues_df = pd.DataFrame(issues)
    display(issues_df)
    issues_df.to_csv(OUT_DIR / "semantic_mep_coordination_issues.csv", index=False)
    print("Saved:", OUT_DIR / "semantic_mep_coordination_issues.csv")
except Exception:
    print(json.dumps(issues, indent=2))
    with open(OUT_DIR / "semantic_mep_coordination_issues.json", "w", encoding="utf-8") as f:
        json.dump(issues, f, indent=2)
    print("Saved:", OUT_DIR / "semantic_mep_coordination_issues.json")

,issue_type,severity,element,element_index,ifc_type,ontology_class,brick_class,message,architectural_element,architectural_index,architectural_ifc_type,edge_index,predicate
0,missing_penetration_or_opening,high,87,5855,None,brick:Equipment,brick:Equipment,MEP distribution element intersects an archite...,63.0,1846.0,NaN,NaN,NaN
1,missing_penetration_or_opening,high,122,13480,None,brick:Equipment,brick:Equipment,MEP distribution element intersects an archite...,63.0,1846.0,NaN,NaN,NaN
2,missing_penetration_or_opening,high,110,28466,None,brick:Equipment,brick:Equipment,MEP distribution element intersects an archite...,63.0,1846.0,NaN,NaN,NaN
3,terminal_not_located_in_space,medium,21,21,None,brick:Terminal_Unit,brick:Terminal_Unit,Terminal/equipment has no derived architectura...,NaN,NaN,NaN,NaN,NaN
4,terminal_not_located_in_space,medium,22,22,None,brick:Terminal_Unit,brick:Terminal_Unit,Terminal/equipment has no derived architectura...,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
706,overasserted_flow_direction,high,1458,29814,None,top:Port,None,IfcRelConnectsPorts was mapped directly to bri...,NaN,NaN,NaN,23954.0,brick:feeds
707,overasserted_flow_direction,high,1460,29816,None,top:Port,None,IfcRelConnectsPorts was mapped directly to bri...,NaN,NaN,NaN,23955.0,brick:feeds
708,overasserted_flow_direction,high,1462,29818,None,top:Port,None,IfcRelConnectsPorts was mapped directly to bri...,NaN,NaN,NaN,23956.0,brick:feeds
709,overasserted_flow_direction,high,1464,29820,None,top:Port,None,IfcRelConnectsPorts was mapped directly to bri...,NaN,NaN,NaN,23957.0,brick:feeds


Saved: mep_space_coordination_outputs\semantic_mep_coordination_issues.csv


## 12. Convert the coordination graph to RDF / KnowledgeGraph

The recent edge-predicate fix means semantic graph edges can now emit direct RDF triples such as:

```turtle
inst:Port_A top:connectsPort inst:Port_B .
inst:Building bot:containsElement inst:Wall_12 .
inst:Duct_21 top:intersects inst:Wall_08 .
inst:Duct_21 top:hasMissingOpening inst:Wall_08 .
```

The edge object can still remain an instance of `top:Relationship`, but its `ontology_predicate` becomes the predicate in RDF.

In [ ]:
triples = TGraph.OntologyTriples(
    coord_graph,
    includeVertices=True,
    includeEdges=True,
    includeDictionaries=True,
    includeBOT=True,
    namespacePrefix="inst",
)

kg = KnowledgeGraph(triples=triples, useRDFLib=True, silent=True)
print("KnowledgeGraph:", kg)
print("Triple count:", len(kg))

# Export Turtle.
ttl_path = OUT_DIR / "mep_space_coordination_graph.ttl"
try:
    kg.ExportToTurtle(str(ttl_path))
except Exception:
    # Fallback for older API names.
    try:
        text = kg.Turtle()
        ttl_path.write_text(text, encoding="utf-8")
    except Exception as exc:
        print("Could not export Turtle:", exc)
else:
    print("Saved:", ttl_path)

## 13. Query examples

These are example SPARQL queries for the exported graph. Depending on your local `KnowledgeGraph` API, you can execute them directly or copy them into an RDFLib graph.

In [ ]:
queries = {
    "MEP elements with missing openings": r'''
PREFIX top: <http://w3id.org/topologicpy#>
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?mep ?barrier ?mepLabel ?barrierLabel
WHERE {
  ?mep top:hasMissingOpening ?barrier .
  OPTIONAL { ?mep rdfs:label ?mepLabel . }
  OPTIONAL { ?barrier rdfs:label ?barrierLabel . }
}
''',
    "MEP elements located in spaces": r'''
PREFIX top: <http://w3id.org/topologicpy#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?mep ?space ?mepLabel ?spaceLabel
WHERE {
  ?mep top:locatedIn ?space .
  OPTIONAL { ?mep rdfs:label ?mepLabel . }
  OPTIONAL { ?space rdfs:label ?spaceLabel . }
}
''',
    "Port connections that are not asserted as Brick flow": r'''
PREFIX top: <http://w3id.org/topologicpy#>
PREFIX brick: <https://brickschema.org/schema/Brick#>

SELECT ?a ?b
WHERE {
  ?a top:connectsPort ?b .
  FILTER NOT EXISTS { ?a brick:feeds ?b . }
}
''',
}

for name, query in queries.items():
    print("\n---", name, "---")
    print(query.strip())

In [ ]:
# Optional execution if KnowledgeGraph exposes a SPARQL/Query method.
def try_query(kg, query):
    for method_name in ["SPARQL", "Query", "Sparql", "query"]:
        method = getattr(kg, method_name, None)
        if callable(method):
            try:
                return method(query)
            except Exception as exc:
                print(f"{method_name} failed:", exc)
    # RDFLib fallback.
    rg = getattr(kg, "_rdf_graph", None)
    if rg is not None:
        try:
            return list(rg.query(query))
        except Exception as exc:
            print("RDFLib query failed:", exc)
    return None

for name, query in queries.items():
    print("\n===", name, "===")
    result = try_query(kg, query)
    if result is None:
        print("Query execution not available in this environment. Query text shown above.")
    else:
        pprint(result)

## 14. Optional: infer `brick:feeds` from port connections

This section intentionally does **not** assert `brick:feeds` from `IfcRelConnectsPorts` alone. Instead, it shows where a future rule should be placed.

A defensible `brick:feeds` assertion needs at least one of the following:

- explicit IFC port flow direction;
- system direction metadata;
- ordered path through a flow network;
- known upstream/downstream equipment types;
- user-confirmed flow convention.

When such evidence exists, add an inferred edge:

```python
ontology_predicate = "brick:feeds"
inferred = True
confidence = ...
reason = "Flow direction inferred from ..."
```

In [ ]:
def infer_brick_feeds_from_ports_when_direction_known(graph):
    """
    Placeholder for a stricter flow-direction reasoner.

    This function intentionally avoids inferring brick:feeds unless there is
    explicit directional evidence in the edge or endpoint dictionaries.
    """
    inferred = []
    for e in active_edges(graph):
        d = ed(e)
        if d.get("ontology_predicate") != "top:connectsPort":
            continue
        # Example directional hints. Adjust these to match your IFC importer output.
        direction = norm_text(d.get("flow_direction") or d.get("FlowDirection") or d.get("direction"))
        if direction in {"source_to_sink", "src_to_dst", "forward", "outlet_to_inlet"}:
            inferred.append(add_inferred_edge(
                graph, e["src"], e["dst"],
                predicate="brick:feeds",
                inverse_predicate="brick:isFedBy",
                relationship="feeds",
                confidence=0.9,
                reason="Flow direction inferred from explicit port direction metadata.",
                evidence="port_flow_direction",
            ))
    return inferred

feeds_edges = infer_brick_feeds_from_ports_when_direction_known(coord_graph)
print("Inferred brick:feeds edges:", len(feeds_edges))

## 15. Final narrative: what this use case demonstrates

This notebook demonstrates a coordination workflow that is stronger than ordinary clash detection:

1. **Federation** discovers all IFC files in a folder and builds one TGraph per source file.
2. **Provenance** preserves the source file, role, graph index, vertex index, and edge index for every imported assertion.
3. **Geometry** detects that something intersects or is located inside something else.
4. **Architecture semantics** identifies whether the object is a space, wall, slab, opening, storey, or building.
5. **MEP semantics** identifies whether the object is a duct, pipe, cable, terminal, luminaire, outlet, or equipment using Brick-aligned classes.
6. **Relationship predicates** preserve IFC relationship meaning, such as `top:connectsPort`, `bot:containsElement`, `top:hasOpening`, and `top:hasPropertySet`.
7. **Reasoning** derives coordination issues such as missing openings and overasserted flow direction.

The resulting graph can be queried, exported as Turtle, and used as the basis for a visual proof graph or dashboard.
